
# 2 individuali uzduotis

Autorius: Augustas Kniška
LSP: 2312054
Variantas: EfficientNet B0; klases: Bee, Castle, Train
Uzdoties versija: 2026-03-17

In [7]:
import sys
!{sys.executable} -m pip install -U scikit-learn seaborn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -----------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C


In [8]:
# import sys
# !{sys.executable} -m pip install seaborn

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.transforms import transforms
from PIL import Image
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   --------------------------------------

In [9]:
data_dir   = 'data'
classes    = ['Train', 'Bee', 'Castle']
num_epochs = 10
batch_size = 32
lr         = 1e-4
test_split = 0.2
seed       = 42
num_workers = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Naudojamas irenginys: {device}")

Naudojamas irenginys: cpu


In [10]:
class ImageFolderDataset(Dataset):
    """Nuskaito vaizdus is aplanko struktura: data/<class>/images/*.jpg"""

    def __init__(self, root_dir, classes, transform=None, cache=False):
        self.root_dir  = root_dir
        self.classes   = classes
        self.transform = transform
        self.cache     = cache
        self._cache    = {}
        self.samples   = []

        for class_idx, class_name in enumerate(classes):
            class_dir = os.path.join(root_dir, class_name.lower(), "images")
            if not os.path.isdir(class_dir):
                print(f"[WARN] Nerasta: {class_dir}")
                continue
            files = glob.glob(os.path.join(class_dir, "*.jpg"))
            for fp in files:
                self.samples.append((fp, class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        if self.cache and file_path in self._cache:
            return self._cache[file_path], label
        image = Image.open(file_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        if self.cache:
            self._cache[file_path] = image
        return image, label

## Duomenų paruošimas

- Duomenų rinkinys padalinamas į **mokymo** (80%) ir **testavimo** (20%) aibes naudojant `random_split`.
- Mokymui naudojamos papildomos augmentacijos (apvertimas, spalvų jitter), testavimui — tik standartinis EfficientNet B0 preprocessing.
- `cache=True` naudojamas testavimo aibei, nes ji perskaitoma tik vieną kartą.

In [11]:
weights = EfficientNet_B0_Weights.DEFAULT
base_preprocess = weights.transforms()

# Papildomos augmentacijos mokymui
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    base_preprocess,
])

full_dataset = ImageFolderDataset(data_dir, classes, transform=None, cache=False)
print(f"Viso vaizdu: {len(full_dataset)}")

# Padalijimas
n_test  = int(len(full_dataset) * test_split)
n_train = len(full_dataset) - n_test
generator = torch.Generator().manual_seed(seed)
train_subset, test_subset = random_split(full_dataset, [n_train, n_test], generator=generator)
print(f"Mokymo aibe: {n_train}, Testavimo aibe: {n_test}")

# Wrapper su skirtingomis transformacijomis
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        file_path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = Image.open(file_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = TransformSubset(train_subset, train_transform)
test_dataset  = TransformSubset(test_subset,  base_preprocess)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)

Viso vaizdu: 2246
Mokymo aibe: 1797, Testavimo aibe: 449


## Modelis

- Naudojamas **EfficientNet-B0** su iš anksto išmokytais svoriais (`EfficientNet_B0_Weights.DEFAULT`).
- Pakeičiamas paskutinis `classifier[1]` sluoksnis, kad atitiktų 3 klases.
- Mokymo metu atnaujinami **visi** modelio parametrai (angl. full fine-tuning).
- Naudojamas `CrossEntropyLoss` ir `Adam` optimizatorius.

In [12]:
model = efficientnet_b0(weights=weights)

# Pakeiciamas paskutinis klasifikavimo sluoksnis
# EfficientNet naudoja model.classifier[1], o ne model.fc
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\kniskiukas/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 14.2MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)


In [ ]:
train_losses = []
train_accs   = []

for epoch in range(num_epochs):
    model.train()
    running_loss    = 0.0
    correct         = 0
    total           = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    print(f"Epoha [{epoch+1:2d}/{num_epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

print("Mokymas baigtas.")

Epoha [ 1/10]  Loss: 0.4295  Acc: 0.9060
Epoha [ 2/10]  Loss: 0.0725  Acc: 0.9883


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, num_epochs+1), train_losses, marker='o', color='steelblue')
axes[0].set_title("Mokymo nuostolis (Loss)")
axes[0].set_xlabel("Epoha")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, num_epochs+1), train_accs, marker='o', color='seagreen')
axes[1].set_title("Mokymo tikslumas (Accuracy)")
axes[1].set_xlabel("Epoha")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#lol

## Testavimas ir metrikos

Testavimo aibėje skaičiuojame:
- **Klasifikavimo matrica** (confusion matrix)
- **Tikslumas** (accuracy)
- **Precizija** (precision)
- **Atkūrimas** (recall)
- **F1** (F1-score)

Visos metrikos skaičiuojamos tiek kiekvienai klasei atskirai, tiek makro vidurkiu.

In [ ]:
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# --- Metrikos ---
accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
recall    = recall_score(all_labels, all_preds, average=None, zero_division=0)
f1        = f1_score(all_labels, all_preds, average=None, zero_division=0)

print(f"{'Klase':<10} {'Precizija':>10} {'Atkurimas':>10} {'F1':>8}")
print("-" * 42)
for i, cls in enumerate(classes):
    print(f"{cls:<10} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>8.4f}")

print("-" * 42)
print(f"{'Makro':<10} {precision.mean():>10.4f} {recall.mean():>10.4f} {f1.mean():>8.4f}")
print(f"\nBendras tikslumas: {accuracy:.4f}")

print("\nISAMUS PRANESIMAS:")
print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes,
    ax=ax,
)
ax.set_xlabel("Numatyta klase")
ax.set_ylabel("Tikroji klase")
ax.set_title("Klasifikavimo matrica (Confusion Matrix)")
plt.tight_layout()
plt.show()

In [ ]:
save_path = os.path.join('models', 'resnet50_finetuned.pth')
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f"Modelis issaugotas: {save_path}")